In [7]:
import pandas as pd
import openpyxl as px
import matplotlib.pyplot as plt
import requests as req

In [8]:
import pandas as pd

df = pd.read_csv("../data/numeros_tva.csv", dtype={"numero_tva": "string"})
print(df.head())

   id                 raison_sociale pays_declare       numero_tva  \
0   1         Nord Technologies GmbH           DK       DK73224645   
1   2      Solaris Technologies SARL           PT      PT787327239   
2   3            Verso Consulting NV           LU       LU71043234   
3   4  Lumen Distribution Sp. z o.o.           SE  SE.751298146001   
4   5             Textil Services Oy           NL   NL505862176B88   

  date_saisie       source_saisie  
0  2025-07-27      portail_client  
1  2024-03-29         reprise_erp  
2  2024-02-16                 crm  
3  2024-02-27  import_fournisseur  
4  2025-04-14                 crm  


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              10000 non-null  int64 
 1   raison_sociale  10000 non-null  str   
 2   pays_declare    10000 non-null  str   
 3   numero_tva      9854 non-null   string
 4   date_saisie     10000 non-null  str   
 5   source_saisie   10000 non-null  str   
dtypes: int64(1), str(4), string(1)
memory usage: 468.9 KB


In [10]:
df["pays_declare"].unique()

<StringArray>
['DK', 'PT', 'LU', 'SE', 'NL', 'BE', 'FI', 'GB', 'FR', 'ZZ', 'PL', 'IT', 'UK',
 'XX', 'QQ']
Length: 15, dtype: str

In [11]:
df["numero_tva"].unique()

<StringArray>
[       'DK73224645',       'PT787327239',        'LU71043234',
   'SE.751298146001',    'NL505862176B88',        'DK61530843',
 '  NL505862176B88 ',   'SE.314125231601',        'DK48646344',
      'BE1997422119',
 ...
      'BE0667627145',    'se342108454601',  'SE 6041 26160001',
      'LU 8783 8948',        'ZZ58038776',       'PT772064113',
       'PT714332224',     'SE96633532830',       'LU.98154452',
        'FI53300336']
Length: 9608, dtype: string

In [12]:
df.isnull().sum()

id                  0
raison_sociale      0
pays_declare        0
numero_tva        146
date_saisie         0
source_saisie       0
dtype: int64

In [13]:
df = pd.read_csv("../data/numeros_tva.csv", dtype={"numero_tva": "string"})

placeholders = {"", "none", "null", "-", "n/a", "na", "nu.ll"}
values = df["numero_tva"].str.strip()

missing_or_placeholder = values.isna() | values.str.casefold().isin(placeholders)
df.loc[missing_or_placeholder, "numero_tva"] = pd.NA

print("Missing or placeholder values:", df["numero_tva"].isna().sum())


Missing or placeholder values: 261


In [14]:
crm = df.loc[
    df["source_saisie"].astype("string").str.strip().str.casefold().eq("crm").fillna(False)
]

pays = crm["pays_declare"].astype("string").str.strip().str.upper()
numero = (
    crm["numero_tva"].astype("string")
    .str.strip()
    .str.upper()
    .str.replace(r"[^A-Z0-9]", "", regex=True)
)

prefixe = numero.str[:2]

# Exclure les pays ou numéros manquants et les valeurs manifestement invalides
placeholders = {"", "NONE", "NULL", "NA", "NAN", "NOTAPPLICABLE"}
valide = (
    pays.notna()
    & pays.ne("")
    & numero.str.match(r"^[A-Z]{2}", na=False)
    & ~numero.isin(placeholders)
)

# Exception : le préfixe TVA grec est EL, alors que le code pays est souvent GR
pays_a_comparer = pays.replace({"GR": "EL"})

correspond = prefixe[valide].eq(pays_a_comparer[valide])
ecarts = crm.loc[valide & ~correspond].copy()

print(f"Lignes CRM comparées : {valide.sum()}")
print(f"Correspondances : {correspond.sum()}")
print(f"Écarts : {(~correspond).sum()}")
print(ecarts[["pays_declare", "numero_tva"]].head(20))

Lignes CRM comparées : 1839
Correspondances : 1839
Écarts : 0
Empty DataFrame
Columns: [pays_declare, numero_tva]
Index: []


In [15]:
data = df.copy()
data["date_saisie"] = pd.to_datetime(data["date_saisie"], errors="coerce")

resume = (
    data.dropna(subset=["date_saisie", "source_saisie"])
        .groupby("source_saisie")
        .agg(
            nombre=("date_saisie", "size"),
            premiere_saisie=("date_saisie", "min"),
            derniere_saisie=("date_saisie", "max"),
            nombre_de_mois=("date_saisie", lambda dates: dates.dt.to_period("M").nunique())
        )
        .sort_values("nombre", ascending=False)
)

print(resume)

                    nombre premiere_saisie derniere_saisie  nombre_de_mois
source_saisie                                                             
reprise_erp           2071      2024-01-01      2025-10-01              22
crm                   2019      2024-01-01      2025-10-01              22
portail_client        1990      2024-01-01      2025-10-01              22
saisie_manuelle       1976      2024-01-01      2025-10-01              22
import_fournisseur    1944      2024-01-01      2025-10-01              22


In [16]:
data = data.dropna(subset=["date_saisie", "source_saisie"]).copy()
data["mois"] = data["date_saisie"].dt.to_period("M")

par_mois = pd.crosstab(data["source_saisie"], data["mois"])
proportions = par_mois.div(par_mois.sum(axis=1), axis=0).round(2)

print(proportions)

mois                2024-01  2024-02  2024-03  2024-04  2024-05  2024-06  \
source_saisie                                                              
crm                    0.04     0.05     0.05     0.06     0.04     0.06   
import_fournisseur     0.05     0.05     0.05     0.05     0.05     0.05   
portail_client         0.05     0.04     0.05     0.04     0.05     0.04   
reprise_erp            0.05     0.04     0.05     0.05     0.05     0.05   
saisie_manuelle        0.04     0.04     0.05     0.05     0.05     0.05   

mois                2024-07  2024-08  2024-09  2024-10  ...  2025-01  2025-02  \
source_saisie                                           ...                     
crm                    0.04     0.05     0.05     0.05  ...     0.05     0.04   
import_fournisseur     0.04     0.05     0.05     0.05  ...     0.05     0.04   
portail_client         0.04     0.05     0.04     0.04  ...     0.05     0.05   
reprise_erp            0.05     0.05     0.05     0.04  ...   

In [ ]:
# le nombre de chiffres de tva (sans prefix)

nb_chiffre_tva = (chiffre_tva.notna() & ~chiffre_tva.str.match(r"^[A-Z]{2}", case=False, na=False)).sum()
mask = (chiffre_tva.notna() & ~chiffre_tva.str.match(r"^[A-Z]{2}", case=False, na=False))
print(nb_chiffre_tva)
print(chiffre_tva[mask].head())

532
119     58560322811
134    396904956B68
162     38486825359
180     33139885439
188     88838354509
Name: numero_tva, dtype: string
